# Data Size Impact Study
**Course**: CS 3402

This study investigates how the size of training data influences the performance of Classical ML models versus Artificial Neural Networks (ANNs).

### Data Download Instructions
Please download the following datasets from Kaggle and place the CSV files in the same directory as this notebook (or in a subfolder named `data/`).

1. **Credit Card Approval**: [Rikdifos Dataset](https://www.kaggle.com/datasets/rikdifos/credit-card-approval-prediction)
   * Need: `application_record.csv` and `credit_record.csv`.
2. **Student Placement**: [Sehaj Dataset](https://www.kaggle.com/datasets/sehaj1104/student-placement-prediction-dataset-2026)
   * Need: `college_student_placement_dataset.csv`.
3. **Digital Distraction**: [Neurocipher Dataset](https://www.kaggle.com/datasets/neurocipher/digital-distraction-vs-academic-performance)
   * Need: CSV file (e.g., `DigitalDistraction.csv` or similar).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.metrics import accuracy_score, mean_squared_error
import os
import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading and Preprocessing

In [ ]:
LIMIT_RECORDS = 5000

def find_path(filename):
    if os.path.exists(filename): return filename
    alt = os.path.join('data', filename)
    if os.path.exists(alt): return alt
    return None

def find_by_keywords(keywords):
    for folder in ['.', 'data']:
        if not os.path.exists(folder): continue
        for f in os.listdir(folder):
            if all(kw.lower() in f.lower() for kw in keywords) and f.endswith('.csv'):
                return os.path.join(folder, f)
    return None

def identify_target(df, keywords, default):
    """Finds a column name containing any of the keywords."""
    for col in df.columns:
        if any(kw.lower() in col.lower() for kw in keywords):
            return col
    return default

datasets = {}
print("Searching for datasets...")

# 1. Credit Card (Classification)
app_path = find_path('application_record.csv')
credit_path = find_path('credit_record.csv')
if app_path and credit_path:
    print("Found Credit Card dataset.")
    app_df = pd.read_csv(app_path, nrows=LIMIT_RECORDS)
    credit_df = pd.read_csv(credit_path)
    credit_df['target'] = credit_df['STATUS'].apply(lambda x: 1 if x in ['1', '2', '3', '4', '5'] else 0)
    target_agg = credit_df.groupby('ID')['target'].max().reset_index()
    datasets['Credit Card'] = {'df': pd.merge(app_df, target_agg, on='ID', how='inner'), 
                             'task': 'classification', 'target': 'target'}
else:
    print("MISSING: Credit Card dataset")

# 2. Student Placement (Classification)
student_file = find_by_keywords(['placement'])
if student_file:
    print(f"Found Student Placement dataset: {student_file}")
    df = pd.read_csv(student_file, nrows=LIMIT_RECORDS)
    target = identify_target(df, ['placed', 'placement', 'status'], 'Placement')
    print(f"  Using target column: {target}")
    datasets['Student Placement'] = {'df': df, 'task': 'classification', 'target': target}
else:
    print("MISSING: Student Placement dataset")

# 3. Digital Distraction (Regression)
dist_file = find_by_keywords(['distraction'])
if dist_file:
    print(f"Found Digital Distraction dataset: {dist_file}")
    df = pd.read_csv(dist_file, nrows=LIMIT_RECORDS)
    target = identify_target(df, ['score', 'final', 'exam', 'performance'], 'final_exam_score')
    print(f"  Using target column: {target}")
    datasets['Digital Distraction'] = {'df': df, 'task': 'regression', 'target': target}
else:
    print("MISSING: Digital Distraction dataset")

## 2. Exploratory Data Analysis (EDA)

In [ ]:
for name, data in datasets.items():
    print(f"\n-- EDA for {name} --")
    display(data['df'].head(3))
    print(f"Loaded Samples: {len(data['df'])}")

## 3. Pipeline Setup

In [ ]:
def create_pipeline(X, model):
    num_cols = X.select_dtypes(include=['int64', 'float64']).columns
    cat_cols = X.select_dtypes(exclude=['int64', 'float64']).columns
    
    preprocessor = ColumnTransformer(transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ])
    return Pipeline([('pre', preprocessor), ('m', model)])

## 4. Experiment Loop

In [ ]:
results = []
fractions = [0.1, 0.3, 0.5, 1.0]
seeds = [42, 123, 999]

for name, info in datasets.items():
    print(f"\nRunning Experiment: {name}...")
    df = info['df'].dropna()
    target = info['target']
    
    if target not in df.columns:
        print(f"  ERROR: Target '{target}' not found in {name}. Available columns: {list(df.columns)}")
        continue
        
    X = df.drop(columns=[target, 'ID'], errors='ignore')
    y = df[target]
    
    X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    if info['task'] == 'classification':
        models = {'Logistic Regression': LogisticRegression(max_iter=500), 
                  'MLP (ANN)': MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, early_stopping=True)}
    else:
        models = {'Ridge': Ridge(), 
                  'MLP (ANN)': MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=300, early_stopping=True)}
                  
    for m_name, model in models.items():
        print(f"  Training {m_name}...")
        for frac in fractions:
            scores = []
            for seed in seeds:
                if frac < 1.0:
                    X_sub, _, y_sub, _ = train_test_split(X_train_full, y_train_full, train_size=frac, random_state=seed)
                else:
                    X_sub, y_sub = X_train_full, y_train_full
                
                pipe = create_pipeline(X, model)
                pipe.fit(X_sub, y_sub)
                preds = pipe.predict(X_test)
                
                if info['task'] == 'classification':
                    scores.append(accuracy_score(y_test, preds))
                else:
                    scores.append(np.sqrt(mean_squared_error(y_test, preds)))
            
            results.append({
                'Dataset': name, 'Model': m_name, 'Fraction': frac, 
                'Samples': len(X_sub), 'Mean': np.mean(scores), 'Std': np.std(scores), 'Task': info['task']
            })

results_df = pd.DataFrame(results)

## 5. Visualization

In [ ]:
if not results_df.empty:
    for name in results_df['Dataset'].unique():
        sub = results_df[results_df['Dataset'] == name]
        plt.figure(figsize=(8, 4))
        for m in sub['Model'].unique():
            m_sub = sub[sub['Model'] == m]
            plt.errorbar(m_sub['Samples'], m_sub['Mean'], yerr=m_sub['Std'], label=m, marker='o')
        plt.title(f"Learning Curve: {name}")
        plt.xlabel("Training Samples"); plt.ylabel("Metric (High better for Class., Low better for Regr.)")
        plt.legend(); plt.grid(True); plt.show()